# Building a complete GPT-style Transformer

This notebook inspects SwiGLU, the complete pre-norm Transformer block, parameter counts, and next-token logits.

In [ ]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(7)

## SwiGLU keeps the sequence shape

The intermediate dimension expands, gates features with SiLU, and projects back to the hidden size.

In [ ]:
from src.model.ffn import SwiGLUFFN

ffn = SwiGLUFFN(dim=64, hidden_dim=128)
x = torch.randn(2, 8, 64)
y = ffn(x)
print("input:", x.shape)
print("output:", y.shape)
assert y.shape == x.shape

## Forward pass through the complete model

The model now includes embeddings, RoPE attention, SwiGLU feed-forward layers, residual connections, final RMSNorm, and the vocabulary head.

In [ ]:
from src.model import DeepSeekConfig, DeepSeekSkeleton

config = DeepSeekConfig(
    vocab_size=256,
    dim=64,
    n_layers=2,
    n_heads=4,
    inter_dim=128,
    max_seq_len=32,
)
model = DeepSeekSkeleton(config)
input_ids = torch.randint(0, config.vocab_size, (2, 8))
logits = model(input_ids)

print("input:", input_ids.shape)
print("logits:", logits.shape)

In [ ]:
total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"parameters: {total_params:,}")

loss = logits.square().mean()
loss.backward()
assert all(parameter.grad is not None for parameter in model.parameters())
print("backward check passed")